# Assignment 3



- Import Libraries

In [ ]:
# Loading Libraries
import nltk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from bs4 import BeautifulSoup
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from gensim.models import Word2Vec
from gensim.models import FastText
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE

nltk.download('stopwords')
nltk.download('wordnet') 

print("--"*50)
print("Libraries loaded successfully.")
print("--"*50)

----------------------------------------------------------------------------------------------------
Libraries loaded successfully.
----------------------------------------------------------------------------------------------------


[nltk_data] Downloading package stopwords to /home/jaee/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /home/jaee/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Part A

- Load dataset

In [31]:
# Loading the dataset
data = pd.read_csv("../data/dataset_with_assignments.csv")

print("--"*50)
print("First 5 Rows of the Dataset:")
print("--"*50)
print(data.head(5))

# Dataset Information
print("--"*50)
print("Dataset Information")
print("--"*50)
print(data.info())

# Checking for missing values
print("--"*50)
print("Missing Values in Each Column:")
print("--"*50)
print(data.isnull().sum())


----------------------------------------------------------------------------------------------------
First 5 Rows of the Dataset:
----------------------------------------------------------------------------------------------------
   page_id                                                url  \
0        1                   http://0769sme.org/index-16.html   
1        4  http://aastocks.com/en/cnhk/quote/quick-quote....   
2        6  http://ada.untergrund.net/?p=boardthread&id=18...   
3        7          http://adrienedurand.wikidot.com/blog:127   
4        8  http://afrafrontpagenews.blogspot.com/2012/03/...   

                           domain  tld                  date  word_count  \
0                     0769sme.org  org  2025-12-04T20:51:02Z        2377   
1                    aastocks.com  com  2025-12-04T21:23:33Z        1738   
2              ada.untergrund.net  net  2025-12-04T20:53:54Z        3759   
3       adrienedurand.wikidot.com  com  2025-12-04T21:11:53Z        1328  

- Preprocessing

In [32]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    # Remove \n, \r, \t
    text = re.sub(r'[\n\r\t]+', ' ', text)

    # Remove HTML tags like <body>, but first remove script/style content
    soup = BeautifulSoup(text, 'html.parser')
    for tag in soup(['script', 'style', 'noscript']):
        tag.decompose()
    text = soup.get_text(separator=' ')
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'(?:https?://|ftp://|www\.)\S+|mailto:\S+|tel:\S+', '', text)
    
    # Remove emails
    text = re.sub(r'\b[a-z0-9._%+-]+@[a-z0-9.-]+\.[a-z]{2,}\b', '', text)
    
    # Remove non-ASCII characters
    text = re.sub(r'[^\x00-\x7F]', ' ', text)
    
    # Keep only letters and spaces
    text = re.sub(r'[^a-z\s]', '', text)

    # Tokenize and filter with stop words and lemmatization
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if len(t) > 2 and t not in stop_words]
    
    return tokens

In [33]:
# Apply preprocessing
sentences = data['full_text'].apply(preprocess).tolist()
print("--"*50)
print("Preprocessing completed. Sample of cleaned text:")
print("--"*50)
print(sentences[0][:5])  # Print first 5 tokens of the first document

/tmp/ipykernel_18644/1682726302.py:9: XMLParsedAsHTMLWarning: It looks like you're using an HTML parser to parse an XML document.

Assuming this really is an XML document, what you're doing might work, but you should know that using an XML parser will be more reliable. To parse this document as XML, make sure you have the Python package 'lxml' installed, and pass the keyword argument `features="xml"` into the BeautifulSoup constructor.

If you want or need to use an HTML parser on this document, you can make this warning go away by filtering it. To do that, run this code before calling the BeautifulSoup constructor:

    from bs4 import XMLParsedAsHTMLWarning
    import warnings

    warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)

  soup = BeautifulSoup(text, 'html.parser')


----------------------------------------------------------------------------------------------------
Preprocessing completed. Sample of cleaned text:
----------------------------------------------------------------------------------------------------
['best', 'resume', 'template', 'ready', 'download']


- Word2Vec Training - CBOW

In [34]:
model_cbow = Word2Vec(
    sentences=sentences,
    vector_size=100,   # embedding dimensions (no need to go high for 6,000 docs)
    window=5,          # context window size (for clustering and topic analysis, avoid too small windows)
    min_count=2,       # ignore words with freq < 2 (Same rationale from previous assignment)
    sg=0,              # 0 = CBOW, 1 = Skip-gram
    workers=4,         # parallel threads
    epochs=15          # training iterations (larger the better, but more computation required)
)


- Word2Vec Training - Skip-gram

In [35]:
model_skipgram = Word2Vec(
    sentences=sentences,
    vector_size=100,   # embedding dimensions (no need to go high for 6,000 docs)
    window=5,          # context window size (for clustering and topic analysis, avoid too small windows)
    min_count=2,       # ignore words with freq < 2 (Same rationale from previous assignment)
    sg=1,              # 0 = CBOW, 1 = Skip-gram
    workers=4,         # parallel threads
    epochs=15          # training iterations (larger the better, but more computation required)
)


- FastText Training - CBOW

In [36]:
model_fasttext_cbow = FastText(
    sentences=sentences,   # same list-of-token-lists from before
    vector_size=100,       # embedding dimensions (no need to go high for 6,000 docs)
    window=5,              # context window size (for clustering and topic analysis, avoid too small windows)
    min_count=2,           # ignore words with freq < 2 (Same rationale from previous assignment)
    sg=0,                  # 0 = CBOW, 1 = Skip-gram
    workers=4,             # parallel threads
    epochs=10,             # training iterations (larger the better, but more computation required)
    min_n=3,               # min char n-gram size
    max_n=6                # max char n-gram size
)

- FastText Training - Skip-gram

In [37]:
model_fasttext_skipgram = FastText(
    sentences=sentences,   # same list-of-token-lists from before
    vector_size=100,       # embedding dimensions (no need to go high for 6,000 docs)
    window=5,              # context window size (for clustering and topic analysis, avoid too small windows)
    min_count=2,           # ignore words with freq < 2 (Same rationale from previous assignment)
    sg=1,                  # 0 = CBOW, 1 = Skip-gram
    workers=4,             # parallel threads
    epochs=10,             # training iterations (larger the better, but more computation required)
    min_n=3,               # min char n-gram size
    max_n=6                # max char n-gram size
)

- Embedding Exploragion

# Part B

- Document Vector Construction

- Classification

- Evaluation and Comparison

- Prediction